# R16c step 7 · Super-resolved crops for the grader: does a sharper picture of small buildings help find-then-grade? (Kaggle T4)

Task **R16c**, step 7: the super-resolution test our supervisor suggested (Research_Directions §8.8, recorded 17 Nov). Built 20 Nov 2026, at Abrar's go. It trains nothing: the three graders of R16c steps 5 and 5b grade the same buildings again, with the crops of small buildings taken from super-resolved photos.

**Why**
- The grader compares two 128 px crops of one building, before and after. Each crop is the building's square window, 32 px wider on every side, enlarged to 128 px by ordinary (bilinear) resizing.
- On xBD's coarser photos most buildings are small: about 85% of SAM 3's outlines there cover under 1,600 px (step 5's saved outlines). Their window is enlarged up to 2×, so the grader sees a blurred building.
- Super-resolution (SR) is a model that enlarges a picture and tries to restore the detail ordinary resizing can't.
- With perfect grades, find-then-grade would reach destroyed IoU 0.556 on validation, against our graders' 0.3066 (mean of three seeds). Most of the loss is in grading.

**What it does, on the 645 validation pairs** (the scores that decide use the 605 clean ones)
- **V0, as before:** crops exactly as R16c step 5 made them. This is the check: it must reproduce the saved grades and scores.
- **S1, Swin2SR:** both photos of a pair are enlarged 2× by Swin2SR (classical ×2; Conde et al., 2022, the successor of SwinIR). It was trained on ordinary photos with a pixel-accuracy loss, so it invents little texture.
  - A building whose longest side is under 48 px takes its crops from the enlarged photos: the same window, at twice the size, shrunk to 128 px.
  - Every other building, and the building-mask channel, stay exactly as in V0.
- **S2, Real-ESRGAN:** the same with Real-ESRGAN ×2 (Wang et al., 2021). It was trained with a GAN loss on photos degraded like real ones (blur, noise, compression), so it is sharper on real photos but invents more texture. That is the risk noted in §8.8: on rubble, invented texture can look like damage or hide it.
- **The same three graders (seeds 42, 43, 44),** with their saved label biases. Two scores, as §8.8 asks:
  1. **find-then-grade's destroyed IoU on the 605 clean pairs:** SAM 3's outlines (E3) filled with their grades, against the truth, exactly as in step 6;
  2. **the graders' macro F1 on the true validation buildings:** every true building (an 8-connected piece of one class, at least 20 px, as in step 5) graded from its crops.

**Checks (the notebook stops if one fails)**
1. V0's log-probabilities match the saved ones within 0.05 on at least 99.5% of SAM 3's outlines.
2. V0's destroyed IoU per seed reproduces 0.3161, 0.3051 and 0.2982: within 0.0005 for seed 42, and within 0.0015 for seeds 43 and 44, as in step 6.
3. V0's macro F1 on the true buildings of all 645 pairs reproduces steps 5 and 5b: 0.7367, 0.7303 and 0.7541, within 0.003. A newer PyTorch build can round a few borderline buildings differently in half precision, as step 6 found.
4. Each SR model, on five real photos (Cell 5):
   - its output, shrunk back to the input's size, matches the input (PSNR above 22 dB), so the colours and the value range are right;
   - Swin2SR beats ordinary bicubic enlargement on photos first shrunk by 2, the task it was trained for.

**The rule (fixed before the run)**
- Of S1 and S2, the one with the higher destroyed IoU (mean of the three seeds) on the 605 clean pairs is the candidate.
- It passes if its gain over V0 is more than noise: the pairs are redrawn 10,000 times, as in E7 step 2b and step 6, and the middle 95% of the gain must lie above 0.
- **If it passes:** a step 7b confirms it once on the 2,007 test pairs, and retraining the grader on super-resolved crops becomes the next candidate.
- **If not:** super-resolution at grading time doesn't help these graders. That is recorded as the answer to the supervisor's suggestion.

**Reported beside it, deciding nothing**
- the macro F1 change, with its own resampled interval, on all true buildings and on the small ones only (where SR acts);
- mIoU, building IoU, and the scores on xBD pairs and on the other sources separately;
- the share of buildings under 48 px, by source;
- `examples.png`: small buildings' crops under V0, S1 and S2, to see by eye whether SR invents texture.

**Why these two models, and the alternatives**
- **Noman's VRT pipeline** (CSE499A, `support/Noman_Super Resolution/`) showed SR on one disaster photo, judged by eye. VRT is a video model: it needs a sequence of frames (the notebook repeats one photo), works only at ×4, and is heavy. Our crops need at most 2× enlargement, from single photos.
- **SwinIR** (2021) is VRT's single-photo relative. **Swin2SR** improves on it and is packaged in the `transformers` library, so it loads with one line.
- **HAT** (2023) scores higher on SR benchmarks, but it is heavier and not packaged. **DiffusionSat** (ICLR 2024) can super-resolve satellite photos, but it is a diffusion model: slow, and it invents more.
- **SR models trained on satellite photos** exist for other resolutions (for example Sentinel-2's 10 m), none for our 0.3–3 m photos. Both models here learned from ordinary photos, which is the risk this test measures.
- **A grader retrained on super-resolved crops** would be the fair test of SR's potential. It costs the graders' training again (about 2 GPU hours per seed, as in steps 5 and 5b), so it comes only if this cheaper test shows promise.

**Expectation (fixed before the run; rough guesses)**
- V0 reproduces the saved scores.
- About 75–90% of the buildings on xBD pairs are under 48 px, and about 30–45% on the other sources.
- The change is small, −0.01 to +0.01 destroyed IoU. The graders were trained on bilinear crops, so sharper crops are new to them, and S2's invented texture may push some intact buildings toward damaged. Neither passes; the real test would need a grader retrained on SR crops.

**How to run on Kaggle** (account 4; the inputs are account 1's, shared with it)
1. Settings: GPU **T4 x2** (both are used for SR), internet **on**: the two SR models are downloaded, Swin2SR from Hugging Face and Real-ESRGAN from its GitHub release. No secret is needed.
2. Inputs:
   - `abrarmohammedtanzim/disasterm3-mirror`;
   - the output of `abrarmohammedtanzim/e3-sam3-building-finding`;
   - the output of `abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines`;
   - the output of `abrarmohammedtanzim/r16c-step5b-third-grader-seed`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (the first 20 validation pairs; it prints the full run's time estimate). Then set it to `False`, and use Save & Run All. About 1–2 h, estimated.

**Outputs** in `/kaggle/working/step7/`:
- `README.md`, `config.json`;
- `scores.csv` (find-then-grade), `true_buildings.csv` (macro F1), `resampling.csv`, `sizes.csv`, `sr_checks.csv`;
- `grades_sr.csv` (every SAM 3 outline's grade per version and seed), `true_grades.csv` (every true building's), `examples.png`.

In [1]:
# ── Cell 1: Environment (Kaggle's own packages; nothing to install) ──────────
import os
import sys

import cv2
import numpy as np
import torch
import torchvision
import transformers

print(f"✓ torch {torch.__version__}, torchvision {torchvision.__version__}, transformers {transformers.__version__}, "
      f"OpenCV {cv2.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(k) for k in range(torch.cuda.device_count())] or 'none (a GPU is needed for speed)'}")

✓ torch 2.11.0+cu130, torchvision 0.26.0+cu130, transformers 5.20.2, OpenCV 4.14.0, python 3.13.16
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd

SMOKE_TEST = False                 # True = the first 20 validation pairs; False = the full run
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/step7")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path("/kaggle/temp")        # Real-ESRGAN's weights; not an output
TEMP.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 4                    # 0 background, 1 intact, 2 damaged, 3 destroyed
LABELS = ["Intact", "Damaged", "Destroyed", "None"]       # the graders' labels
SEEDS = (42, 43, 44)
BIAS_FIXED = {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}
CROP, MARGIN = 128, 32             # as the graders trained (R16c steps 2 and 5)
MIN_AREA = 20                      # px: true building pieces below this are skipped (step 5's rule)
SMALL = 48                         # px: a building whose longest side is under this takes its crops from the SR photos
SR_MODELS = {"S1": "caidas/swin2SR-classical-sr-x2-64",   # Swin2SR, classical x2, on Hugging Face
             "S2": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"}
TILE, TILE_PAD = 512, 16           # px: the photos are super-resolved in tiles, each read with this much extra context
PUBLISHED = {42: 0.3161, 43: 0.3051, 44: 0.2982}          # destroyed IoU, R16c steps 5 and 5b, 605 clean pairs
SCORE_TOL = {42: 0.0005, 43: 0.0015, 44: 0.0015}          # as in step 6
MACRO_PUBLISHED = {42: 0.7367, 43: 0.7303, 44: 0.7541}    # macro F1 on the true buildings, all 645 pairs (steps 5, 5b)
MACRO_TOL = 0.003
LOGP_TOL, LOGP_SHARE = 0.05, 0.995
CONSISTENCY_DB = 22                # Cell 5: an SR output shrunk back must match its input at least this well
N_RESAMPLES, RNG_SEED = 10_000, 0

# E5 step 1's 40 validation pairs that share images with training (val_train_leaks_hash.csv, counted True)
LEAKED = {f"socal_fire_{n:08d}_post_disaster.png" for n in
          (81, 188, 218, 220, 222, 235, 301, 341, 348, 396, 436, 455, 497, 504, 614, 1188)} | \
         {f"woolsey_fire_{n:08d}_post_disaster.png" for n in
          (9, 30, 32, 137, 272, 399, 435, 474, 498, 514, 529, 573, 576, 589, 623, 632, 634, 653, 679, 703, 722, 758,
           783, 810)}
assert len(LEAKED) == 40

FOUND, FOUND_DIRS = defaultdict(list), defaultdict(list)
for root, dirs, files in os.walk(INPUT_ROOT):
    for name in files:
        if name in ("instances_val.csv", "calibration.json") or name.startswith("grader_seed"):
            FOUND[name].append(Path(root) / name)
    for name in dirs:
        if name == "pre_grid2":
            FOUND_DIRS[name].append(Path(root) / name)
WHY = "attach Your Work → r16c-step5-grader-on-sam3-outlines and r16c-step5b-third-grader-seed"
GRADER_PATHS = {}
for s in SEEDS:
    hits = sorted(FOUND[f"grader_seed{s}.pth"])
    if not hits:
        raise FileNotFoundError(f"No grader_seed{s}.pth: {WHY}")
    GRADER_PATHS[s] = hits[0]
BIASES = {}
for path in FOUND["calibration.json"]:
    for seed, chosen in json.loads(path.read_text(encoding="utf-8")).items():
        if int(seed) in SEEDS and [chosen["bias"][l] for l in LABELS] == BIAS_FIXED[int(seed)]:
            BIASES[int(seed)] = np.array(BIAS_FIXED[int(seed)])
assert sorted(BIASES) == list(SEEDS), f"Biases found for {sorted(BIASES)} only: {WHY}"
INST_PATHS = {}
for path in FOUND["instances_val.csv"]:
    cols = set(pd.read_csv(path, nrows=0).columns)
    for s in SEEDS:
        if {f"logp_{l.lower()}_s{s}" for l in LABELS} <= cols:
            INST_PATHS[s] = path
assert sorted(INST_PATHS) == list(SEEDS), f"Saved grades found for {sorted(INST_PATHS)} only: {WHY}"
if not FOUND_DIRS["pre_grid2"]:
    raise FileNotFoundError("No pre_grid2 folder (SAM 3's saved maps): attach Your Work → e3-sam3-building-finding")
E3_DIR = sorted(FOUND_DIRS["pre_grid2"])[0]
assert MANIFEST_PATH.exists(), "attach abrarmohammedtanzim/disasterm3-mirror"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ R16c step 7{'  [SMOKE TEST]' if SMOKE_TEST else ''}: graders {GRADER_PATHS}")
print(f"  SAM 3 maps: {E3_DIR}; saved grades: {INST_PATHS}")

✓ R16c step 7: graders {42: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/grader_seed42.pth'), 43: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/grader_seed43.pth'), 44: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5b-third-grader-seed/step5b/grader_seed44.pth')}
  SAM 3 maps: /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2; saved grades: {44: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5b-third-grader-seed/step5b/instances_val.csv'), 42: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/instances_val.csv'), 43: PosixPath('/kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/instances_val.csv')}


In [3]:
# ── Cell 3: The validation pairs (the D4 split, as in L1c) and the 605 clean ones ──
from sklearn.model_selection import train_test_split

with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)


def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
              DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / filename):
        if c.exists():
            return str(c)
    return None


by_image = defaultdict(dict)       # the D4 notebooks' Cell 4, with the before-photo (R11's version)
for e in raw:
    if e.get("task") != "Referring Expression Segmentation" or e.get("cls_description") != "Building Damage Assessment":
        continue
    if e.get("image_type") != "Optical":
        continue
    post_img = e.get("post_image_path", "")
    folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name
    resolved = resolve_path(e.get("ground_truth", ""))
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")
del raw
FOLDER_TO_CLASS = {"train_building_intact_mask": 1, "train_building_damaged_mask": 2, "train_building_destroyed_mask": 3}
pairs = []
for post_img, d in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path or not any(k in d for k in FOLDER_TO_CLASS):
        continue
    pairs.append({"image_path": img_path, "pre_image_path": resolve_path(d.get("pre_image_path")),
                  "masks": {k: d[k] for k in FOLDER_TO_CLASS if k in d}})
_, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
IS_CLEAN = {Path(p["image_path"]).name: Path(p["image_path"]).name not in LEAKED for p in val_pairs}
assert len(val_pairs) == 645 and sum(IS_CLEAN.values()) == 605, f"{len(val_pairs)} validation pairs, {sum(IS_CLEAN.values())} clean"
assert all(p["pre_image_path"] for p in val_pairs), "a validation pair has no before-photo"
if SMOKE_TEST:
    val_pairs = val_pairs[:20]
    IS_CLEAN = {Path(p["image_path"]).name: IS_CLEAN[Path(p["image_path"]).name] for p in val_pairs}


def truth_of(pair, shape):
    """The D4 rule: intact, then damaged, then destroyed, each written over the last; at the photo's size."""
    truth = None
    for folder, cls in FOLDER_TO_CLASS.items():
        if folder in pair["masks"]:
            m = cv2.imread(pair["masks"][folder], cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            truth = np.zeros_like(m, np.uint8) if truth is None else truth
            truth[m > 0] = cls
    if truth.shape != shape:
        truth = cv2.resize(truth, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return truth


print(f"✓ {len(pairs):,} pairs; validation {len(val_pairs)}, clean {sum(IS_CLEAN.values())}{' (SMOKE TEST: first 20)' if SMOKE_TEST else ''}")

✓ 6,443 pairs; validation 645, clean 605


In [4]:
# ── Cell 4: Helpers: step 6's grader code (unchanged), the SR crops, the tiling, the scoring; with self-checks ──
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage


def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side


def longest_side(sl):
    return max(sl[0].stop - sl[0].start, sl[1].stop - sl[1].start)


def small_ids(labels, ids):
    """The ids whose building's longest side is under SMALL px."""
    slices = ndimage.find_objects(labels)
    return [lid for lid in ids if longest_side(slices[lid - 1]) < SMALL]


def building_crops(pre, post, labels, ids, sr=None):
    """uint8 (len(ids), 7, CROP, CROP): before RGB, after RGB, and 255 on the building; R16c step 5's code. With
    sr = (pre, post) super-resolved 2x, a building whose longest side is under SMALL px is cropped from those instead:
    the same window at twice the size, shrunk to CROP px (INTER_AREA). Every other building, and the mask, as before."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        sl = slices[lid - 1]
        top, left, side = square_box(sl, h, w)
        win = (slice(top, top + side), slice(left, left + side))
        if sr is not None and longest_side(sl) < SMALL:
            win2 = (slice(2 * top, 2 * (top + side)), slice(2 * left, 2 * (left + side)))
            out[n, 0:3] = cv2.resize(sr[0][win2], (CROP, CROP), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
            out[n, 3:6] = cv2.resize(sr[1][win2], (CROP, CROP), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
        else:
            out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
            out[n, 3:6] = cv2.resize(post[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP), interpolation=cv2.INTER_NEAREST)
    return out


def true_pieces(gt):
    """The true buildings of one pair, as in step 5: the 8-connected pieces of each true class of at least MIN_AREA
    px, as (label 0-2, id map, ids)."""
    out = []
    for cls in (1, 2, 3):
        n, labels, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
        ids = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]
        if ids:
            out.append((cls - 1, labels, ids))
    return out


class SiameseGrader(nn.Module):
    """A shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> one score per label. R16c step 5's model."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(256, len(LABELS)))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))


MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255


def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)


@torch.no_grad()
def log_probs(model, crops):
    """uint8 crops (N, 7, S, S) -> log-probabilities (N, labels)."""
    model.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.log_softmax(model(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, len(LABELS)), np.float32)


def load_grader(path):
    model = SiameseGrader()
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    return model.to(DEVICE).eval()


def to_class(logp, seed):
    """Grades with the seed's biases: intact, damaged, destroyed -> 1-3; "None" (not a building) -> 0."""
    label = (logp + BIASES[seed]).argmax(1)
    return np.where(label < 3, label + 1, 0)


# ── Super-resolution: a whole photo, in tiles ──
@torch.no_grad()
def super_resolve(model_fn, img, device=DEVICE, tile=TILE):
    """RGB uint8 (H, W, 3) -> RGB uint8 (2H, 2W, 3), enlarged by an SR model (a function of a (1, 3, h, w) tensor in
    [0, 1]). Each tile is read with TILE_PAD px of context on every side, padded to a multiple of 16, and only its own
    part of the output is kept. Half precision; a tile that overflows is redone in full precision."""
    h, w = img.shape[:2]
    x = torch.from_numpy(np.ascontiguousarray(img)).to(device).permute(2, 0, 1)[None].float() / 255
    out = torch.empty((1, 3, 2 * h, 2 * w), device=device)
    for y0 in range(0, h, tile):
        for x0 in range(0, w, tile):
            y1, x1 = min(y0 + tile, h), min(x0 + tile, w)
            ya, xa, yb, xb = max(y0 - TILE_PAD, 0), max(x0 - TILE_PAD, 0), min(y1 + TILE_PAD, h), min(x1 + TILE_PAD, w)
            patch = x[:, :, ya:yb, xa:xb]
            ph, pw = patch.shape[2:]
            patch = F.pad(patch, (0, (-pw) % 16, 0, (-ph) % 16), mode="reflect")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                sr = model_fn(patch).float()
            if not torch.isfinite(sr).all():
                sr = model_fn(patch).float()                   # outside autocast: full precision
            oy, ox = 2 * (y0 - ya), 2 * (x0 - xa)
            out[:, :, 2 * y0:2 * y1, 2 * x0:2 * x1] = sr[:, :, oy:oy + 2 * (y1 - y0), ox:ox + 2 * (x1 - x0)]
    return (out[0].clamp(0, 1).permute(1, 2, 0) * 255).round().to(torch.uint8).cpu().numpy()


def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float(10 * np.log10(255 ** 2 / max(mse, 1e-10)))


# ── The scoring (as E7 and R16c) ──
def confusion(truth, pred):
    return np.bincount(truth.ravel().astype(np.int64) * NUM_CLASSES + pred.ravel(), minlength=NUM_CLASSES ** 2).reshape(4, 4)


def scores(cm):
    tp = np.diag(cm).astype(np.float64)
    iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)
    b = cm[1:, 1:].sum()
    return {"miou": iou.mean(), "destroyed_iou": iou[3], "building_iou": b / max(b + cm[0, 1:].sum() + cm[1:, 0].sum(), 1)}


def macro_f1_many(c):
    """(R, 3 true grades, 4 labels) -> R macro F1s: the mean F1 of the three grades, "None" counted as wrong (step 5)."""
    c = np.asarray(c, np.float64)
    tp = np.stack([c[:, i, i] for i in range(3)], 1)
    precision = tp / np.maximum(c[:, :, :3].sum(1), 1)
    recall = tp / np.maximum(c.sum(2), 1)
    return (2 * precision * recall / np.maximum(precision + recall, 1e-8)).mean(1)


def macro_f1(cm):
    return float(macro_f1_many(np.asarray(cm)[None])[0])


# Self-checks: SR crops only for small buildings, the masks unchanged; the tiles stitch exactly; the scores
_rng = np.random.default_rng(0)
_img = cv2.GaussianBlur((_rng.random((300, 400, 3)) * 255).astype(np.uint8), (0, 0), 2)
_lab = np.zeros((300, 400), np.int32)
_lab[10:30, 10:40], _lab[100:180, 200:290] = 1, 2         # longest sides 30 px (small) and 90 px
_up = cv2.resize(_img, (800, 600), interpolation=cv2.INTER_CUBIC)     # a stand-in for an SR model
_v0, _s = building_crops(_img, _img, _lab, [1, 2]), building_crops(_img, _img, _lab, [1, 2], sr=(_up, _up))
assert _v0.shape == _s.shape == (2, 7, CROP, CROP) and small_ids(_lab, [1, 2]) == [1]
assert np.array_equal(_v0[1], _s[1]) and np.array_equal(_v0[:, 6], _s[:, 6])
assert not np.array_equal(_v0[0, :6], _s[0, :6]) and np.abs(_v0[0, :6].astype(int) - _s[0, :6]).mean() < 8
_x = (_rng.random((300, 700, 3)) * 255).astype(np.uint8)
_near = super_resolve(lambda t: F.interpolate(t, scale_factor=2, mode="nearest"), _x, tile=128)
assert np.array_equal(_near, _x.repeat(2, 0).repeat(2, 1)), "the tiles don't stitch back exactly"
_gt = np.zeros((200, 300), np.uint8)
_gt[0:10, 0:20], _gt[100:150, 200:260], _gt[190:192, 0:2] = 3, 1, 2      # the 4-pixel piece is below MIN_AREA
assert [(l, len(i)) for l, _, i in true_pieces(_gt)] == [(0, 1), (2, 1)]
assert macro_f1(np.diag([5, 3, 2, 0])[:3]) == 1.0 and macro_f1(np.array([[0, 0, 0, 4]] * 3)) == 0.0
assert to_class(np.array([[-0.1, -9, -9, -9], [-9, -9, -9, -0.1]]), 42).tolist() == [1, 0]
_t = np.array([[0, 3], [3, 1]], np.uint8)
assert abs(scores(confusion(_t, _t))["destroyed_iou"] - 1) < 1e-9
print("✓ Helpers ready (self-checks passed: SR crops for small buildings only, the tiles stitch exactly)")

✓ Helpers ready (self-checks passed: SR crops for small buildings only, the tiles stitch exactly)


In [5]:
# ── Cell 5: The two SR models, one copy per GPU, and their checks on real photos ──────────
import hashlib

from transformers import Swin2SRForImageSuperResolution


class ResidualDenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        return self.conv5(torch.cat((x, x1, x2, x3, x4), 1)) * 0.2 + x


class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.rdb1, self.rdb2, self.rdb3 = ResidualDenseBlock(nf, gc), ResidualDenseBlock(nf, gc), ResidualDenseBlock(nf, gc)

    def forward(self, x):
        return self.rdb3(self.rdb2(self.rdb1(x))) * 0.2 + x


class RRDBNet(nn.Module):
    """Real-ESRGAN's generator (BasicSR's RRDBNet) at x2: the input is pixel-unshuffled by 2, then enlarged 4x."""

    def __init__(self, nf=64, nb=23, gc=32):
        super().__init__()
        self.conv_first = nn.Conv2d(3 * 4, nf, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_hr = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, 3, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(F.pixel_unshuffle(x, 2))
        feat = feat + self.conv_body(self.body(feat))
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        return self.conv_last(self.lrelu(self.conv_hr(feat)))


SR_DEVICES = [torch.device(f"cuda:{k}") for k in range(min(torch.cuda.device_count(), 2))] or [DEVICE]
_swin = [Swin2SRForImageSuperResolution.from_pretrained(SR_MODELS["S1"]).to(d).eval() for d in SR_DEVICES]
assert _swin[0].config.upscale == 2, f"Swin2SR enlarges {_swin[0].config.upscale}x, not 2x"
SWIN_REVISION = getattr(_swin[0].config, "_commit_hash", None)
esr_file = TEMP / Path(SR_MODELS["S2"]).name
if not esr_file.exists():
    torch.hub.download_url_to_file(SR_MODELS["S2"], str(esr_file), progress=False)
ESR_SHA256 = hashlib.sha256(esr_file.read_bytes()).hexdigest()
_state = torch.load(esr_file, map_location="cpu", weights_only=True)
_state = _state.get("params_ema", _state.get("params", _state))
_esr = []
for d in SR_DEVICES:
    m = RRDBNet()
    m.load_state_dict(_state, strict=True)                 # strict: every weight matches the architecture
    _esr.append(m.to(d).eval())
SR_FN = {"S1": [lambda t, m=m: m(pixel_values=t).reconstruction for m in _swin], "S2": _esr}
NAMES = {"S1": "S1 (Swin2SR)", "S2": "S2 (Real-ESRGAN)"}

# Checks on the centre (256 x 256) of five clean pairs' after-photos
rows = []
for p in [p for p in val_pairs if IS_CLEAN[Path(p["image_path"]).name]][:5]:
    photo = cv2.cvtColor(cv2.imread(p["image_path"]), cv2.COLOR_BGR2RGB)
    h, w = photo.shape[:2]
    assert min(h, w) >= 256, (p["image_path"], photo.shape)
    patch = photo[h // 2 - 128:h // 2 + 128, w // 2 - 128:w // 2 + 128]
    half = cv2.resize(patch, (128, 128), interpolation=cv2.INTER_AREA)           # the photo shrunk by 2
    row = {"image": Path(p["image_path"]).name, "bicubic_psnr": round(psnr(cv2.resize(half, (256, 256), interpolation=cv2.INTER_CUBIC), patch), 2)}
    for v, fns in SR_FN.items():
        row[f"{v}_psnr"] = round(psnr(super_resolve(fns[0], half, SR_DEVICES[0]), patch), 2)
        back = cv2.resize(super_resolve(fns[0], patch, SR_DEVICES[0]), (256, 256), interpolation=cv2.INTER_AREA)
        row[f"{v}_consistency_psnr"] = round(psnr(back, patch), 2)
    rows.append(row)
SR_CHECK = pd.DataFrame(rows)
print(SR_CHECK.to_string(index=False))
assert (SR_CHECK[["S1_consistency_psnr", "S2_consistency_psnr"]] > CONSISTENCY_DB).all().all(), \
    "An SR output, shrunk back, doesn't match its input: check the colours and the value range"
assert SR_CHECK["S1_psnr"].mean() > SR_CHECK["bicubic_psnr"].mean(), "Swin2SR doesn't beat bicubic enlargement"

# The time for one whole photo, and the full run's SR time
SR_PHOTO_SECONDS = {}
for v, fns in SR_FN.items():
    t = time.time()
    super_resolve(fns[0], photo, SR_DEVICES[0])
    SR_PHOTO_SECONDS[v] = round(time.time() - t, 2)
full_h = 645 * 2 * sum(SR_PHOTO_SECONDS.values()) / len(SR_DEVICES) / 3600
print(f"✓ SR models ready on {[str(d) for d in SR_DEVICES]}: Swin2SR (revision {SWIN_REVISION}), Real-ESRGAN "
      f"(sha256 {ESR_SHA256[:16]}...). One {w} x {h} photo: {SR_PHOTO_SECONDS} s; the full run's SR, at most about {full_h:.1f} h")

config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 48.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

                                         image  bicubic_psnr  S1_psnr  S1_consistency_psnr  S2_psnr  S2_consistency_psnr
               turkey_earthquake3_post_692.png         40.91    42.24                55.28    26.46                29.23
 hurricane_florence_00000315_post_disaster.png         42.37    42.18                54.74    36.39                35.00
     nepal_flooding_00000278_post_disaster.png         37.16    36.86                52.44    33.13                34.03
santa_rosa_wildfire_00000201_post_disaster.png         27.55    27.09                37.70    25.09                27.52
  hurricane_michael_00000224_post_disaster.png         37.08    37.12                51.51    32.70                36.93
✓ SR models ready on ['cuda:0', 'cuda:1']: Swin2SR (revision None), Real-ESRGAN (sha256 49fafd45f8fd7aa8...). One 1024 x 1024 photo: {'S1': 17.82, 'S2': 1.46} s; the full run's SR, at most about 3.5 h


In [6]:
# ── Cell 6: Every validation pair: the SR photos, then the three graders on its true buildings and SAM 3's outlines ──
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

graders = {s: load_grader(GRADER_PATHS[s]) for s in SEEDS}
saved = {s: pd.read_csv(INST_PATHS[s]).query("found_by == 'sam3'") for s in SEEDS}
VARIANTS = ("V0", "S1", "S2")
LOGP_COLS = {s: [f"logp_{l.lower()}_s{s}" for l in LABELS] for s in SEEDS}
POOL = ThreadPoolExecutor(max_workers=len(SR_DEVICES))


def sr_pair(v, pre, post):
    """Both photos enlarged by SR model v: the before-photo on the first GPU, the after-photo on the second, at once."""
    jobs = [POOL.submit(super_resolve, SR_FN[v][k % len(SR_DEVICES)], img, SR_DEVICES[k % len(SR_DEVICES)])
            for k, img in enumerate((pre, post))]
    return tuple(j.result() for j in jobs)


CMS = {(v, s): {} for v in VARIANTS for s in SEEDS}        # find-then-grade: per clean pair, 4 x 4
TRUE_CMS = {(v, s): {} for v in VARIANTS for s in SEEDS}   # true buildings: per pair, 3 true grades x 4 labels
records, true_records, logp_check, sizes, EXAMPLES = [], [], [], [], []
SR_SECONDS, n_sr = defaultdict(float), 0
t0 = time.time()
for p in tqdm(val_pairs, desc="pairs"):
    name = Path(p["image_path"]).name
    clean = IS_CLEAN[name]
    post = cv2.cvtColor(cv2.imread(p["image_path"]), cv2.COLOR_BGR2RGB)
    pre = cv2.cvtColor(cv2.imread(p["pre_image_path"]), cv2.COLOR_BGR2RGB)
    if pre.shape != post.shape:                           # as E7 and the D4 dataset class
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    truth = truth_of(p, post.shape[:2])
    parts = true_pieces(truth)
    ids, sam = [], None
    if clean:
        sam = cv2.imread(str(E3_DIR / f"{Path(name).stem}.png"), cv2.IMREAD_UNCHANGED)
        sam = np.zeros(post.shape[:2], np.int32) if sam is None else sam.astype(np.int32)
        if sam.shape != post.shape[:2]:
            sam = cv2.resize(sam, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_NEAREST)
        ids = saved[42][saved[42]["image"] == name]["id"].tolist()
    true_small = np.concatenate([np.isin(i, small_ids(lab, i)) for _, lab, i in parts]) if parts else np.zeros(0, bool)
    sam_small = np.isin(ids, small_ids(sam, ids)) if ids else np.zeros(0, bool)
    sizes.append({"image": name, "clean": clean, "true_buildings": len(true_small), "true_small": int(true_small.sum()),
                  "outlines": len(ids), "outlines_small": int(sam_small.sum())})

    # The SR photos, only where some building is small (elsewhere every version's crops are V0's)
    srs = dict.fromkeys(VARIANTS)
    if true_small.any() or sam_small.any():
        for v in ("S1", "S2"):
            t = time.time()
            srs[v] = sr_pair(v, pre, post)
            SR_SECONDS[v] += time.time() - t
        n_sr += 1

    if parts:
        true_labels = np.concatenate([np.full(len(i), l) for l, _, i in parts])
    if ids:
        under = np.bincount(sam.ravel().astype(np.int64) * NUM_CLASSES + truth.ravel(),
                            minlength=(int(sam.max()) + 1) * NUM_CLASSES).reshape(-1, NUM_CLASSES)[np.array(ids)]
        main_truth = under.argmax(1)                      # the outline's main true class (0 = mostly background)
    crops_true = {v: np.concatenate([building_crops(pre, post, lab, i, sr=srs[v]) for _, lab, i in parts])
                  for v in VARIANTS} if parts else {}
    crops_sam = {v: building_crops(pre, post, sam, ids, sr=srs[v]) for v in VARIANTS} if ids else {}
    for s in SEEDS:
        for v in VARIANTS:
            # 1) The true buildings, as in step 5 (rows: the true grade; columns: the label, with the seed's biases)
            cm = np.zeros((3, len(LABELS)), np.int64)
            if parts:
                label = (log_probs(graders[s], crops_true[v]) + BIASES[s]).argmax(1)
                np.add.at(cm, (true_labels, label), 1)
                true_records.extend({"image": name, "true": int(t), "small": bool(sm), "variant": v, "seed": s, "label": int(l)}
                                    for t, sm, l in zip(true_labels, true_small, label))
            TRUE_CMS[(v, s)][name] = cm
            # 2) SAM 3's outlines filled with their grades, as in step 6 (clean pairs only)
            if not clean:
                continue
            classes = np.zeros(0, np.int64)
            if ids:
                lp = log_probs(graders[s], crops_sam[v])
                classes = to_class(lp, s)
                if v == "V0":
                    mine = saved[s][saved[s]["image"] == name].set_index("id").loc[ids, LOGP_COLS[s]].to_numpy()
                    logp_check.append(np.abs(lp - mine).max(1))
            lut = np.zeros(int(sam.max()) + 1, np.uint8)
            if ids:
                lut[np.array(ids)] = classes
            CMS[(v, s)][name] = confusion(truth, lut[sam])
            if ids:
                records.extend({"image": name, "id": i, "variant": v, "seed": s, "grade": int(c), "main_truth": int(t),
                                "small": bool(sm)} for i, c, t, sm in zip(ids, classes, main_truth, sam_small))
    # A few small true buildings, intact and destroyed, under each version, for examples.png
    if srs["S1"] is not None and parts and len(EXAMPLES) < 8:
        for k in np.flatnonzero(true_small):
            grade = {0: "intact", 2: "destroyed"}.get(int(true_labels[k]))
            if grade and sum(e["grade"] == grade for e in EXAMPLES) < 4:
                EXAMPLES.append({"image": name, "grade": grade, **{v: crops_true[v][k] for v in VARIANTS}})
POOL.shutdown()

GRADES = pd.DataFrame(records)
TRUE_GRADES = pd.DataFrame(true_records)
SIZES = pd.DataFrame(sizes)
LOGP_DIFF = np.concatenate(logp_check) if logp_check else np.zeros(0)
MINUTES = (time.time() - t0) / 60
SR_PAIR_SECONDS = {v: round(SR_SECONDS[v] / max(n_sr, 1), 2) for v in ("S1", "S2")}
print(f"✓ {len(val_pairs)} pairs ({sum(IS_CLEAN.values())} clean) in {MINUTES:.1f} min; SR on {n_sr} pairs, "
      f"{SR_PAIR_SECONDS} s per pair (both photos at once)")
if SMOKE_TEST:
    print(f"  The full run's Cell 6, estimated: about {MINUTES / len(val_pairs) * 645 / 60:.1f} h")

pairs:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 645 pairs (605 clean) in 281.2 min; SR on 633 pairs, {'S1': 23.93, 'S2': 1.55} s per pair (both photos at once)


In [7]:
# ── Cell 7: Checks, scores, the rule, resampling, the example sheet and the files ─────────────
import re

IMAGES = [n for n, c in IS_CLEAN.items() if c]             # the clean pairs
ALL = list(IS_CLEAN)
XBD = re.compile(r"^(.+?)_(\d{8})_post_disaster")
GROUPS = {"all": IMAGES, "xBD": [i for i in IMAGES if XBD.match(Path(i).stem)],
          "other sources": [i for i in IMAGES if not XBD.match(Path(i).stem)]}


def total(cms, images, shape=(4, 4)):
    return sum((cms[i] for i in images), np.zeros(shape, np.int64))


def cm_of(df):
    cm = np.zeros((3, len(LABELS)), np.int64)
    np.add.at(cm, (df["true"].to_numpy(int), df["label"].to_numpy(int)), 1)
    return cm


# Check 1: V0's log-probabilities against the saved ones
share_ok = float((LOGP_DIFF <= LOGP_TOL).mean()) if len(LOGP_DIFF) else 1.0
CHECKS = [{"check": f"V0 log-probabilities within {LOGP_TOL} of the saved ones", "result": f"{share_ok:.4f} of outlines",
           "pass": share_ok >= LOGP_SHARE}]
# Check 2: V0's destroyed IoU per seed against R16c's published scores (only meaningful on all 605 clean pairs)
for s in SEEDS:
    d = scores(total(CMS[("V0", s)], IMAGES))["destroyed_iou"]
    CHECKS.append({"check": f"V0 destroyed IoU, seed {s}, against {PUBLISHED[s]} (within {SCORE_TOL[s]})", "result": round(float(d), 4),
                   "pass": SMOKE_TEST or abs(d - PUBLISHED[s]) <= SCORE_TOL[s]})
# Check 3: V0's macro F1 on the true buildings of all 645 pairs against steps 5 and 5b
for s in SEEDS:
    f = macro_f1(total(TRUE_CMS[("V0", s)], ALL, (3, len(LABELS))))
    CHECKS.append({"check": f"V0 macro F1 on the true buildings of all pairs, seed {s}, against {MACRO_PUBLISHED[s]} (within {MACRO_TOL})",
                   "result": round(f, 4), "pass": SMOKE_TEST or abs(f - MACRO_PUBLISHED[s]) <= MACRO_TOL})
CHECKS = pd.DataFrame(CHECKS)
print(CHECKS.to_string(index=False))
if not CHECKS["pass"].all():
    raise RuntimeError("A check failed: V0 doesn't reproduce the saved grades or scores, so S1 and S2 can't be compared. Stop.")

# Find-then-grade, as in step 6 (mean of three seeds), with the grades' agreement on all outlines and on small ones
rows = []
for v in VARIANTS:
    for group, images in GROUPS.items():
        per_seed = [scores(total(CMS[(v, s)], images)) for s in SEEDS]
        g = GRADES[(GRADES["variant"] == v) & GRADES["image"].isin(images)] if len(GRADES) else GRADES
        gs = g[g["small"]] if len(g) else g
        rows.append({"variant": v, "pairs": group, "n_pairs": len(images),
                     **{k: round(float(np.mean([x[k] for x in per_seed])), 4) for k in per_seed[0]},
                     **{f"destroyed_iou_s{s}": round(float(x["destroyed_iou"]), 4) for s, x in zip(SEEDS, per_seed)},
                     "grade_agrees_with_main_truth": round(float((g["grade"] == g["main_truth"]).mean()), 4) if len(g) else float("nan"),
                     "agrees_on_small_outlines": round(float((gs["grade"] == gs["main_truth"]).mean()), 4) if len(gs) else float("nan")})
SCORES = pd.DataFrame(rows)
print(SCORES.to_string(index=False))

# The graders on the true buildings: macro F1 (mean of three seeds) on the clean pairs and on all, and on small ones only
trows = []
for v in VARIANTS:
    for subset, images in (("clean", IMAGES), ("all", ALL)):
        f1s = [macro_f1(total(TRUE_CMS[(v, s)], images, (3, len(LABELS)))) for s in SEEDS]
        t = TRUE_GRADES[(TRUE_GRADES["variant"] == v) & TRUE_GRADES["image"].isin(images) & TRUE_GRADES["small"]] \
            if len(TRUE_GRADES) else TRUE_GRADES
        f1_small = [macro_f1(cm_of(t[t["seed"] == s])) for s in SEEDS] if len(t) else [float("nan")] * len(SEEDS)
        trows.append({"variant": v, "pairs": subset, "buildings": int(total(TRUE_CMS[(v, SEEDS[0])], images, (3, len(LABELS))).sum()),
                      "macro_f1": round(float(np.mean(f1s)), 4), **{f"macro_f1_s{s}": round(f, 4) for s, f in zip(SEEDS, f1s)},
                      "small_buildings": len(t) // len(SEEDS), "macro_f1_small": round(float(np.mean(f1_small)), 4)})
TRUE_SCORES = pd.DataFrame(trows)
print(TRUE_SCORES.to_string(index=False))

# The rule: of S1 and S2, the higher destroyed IoU (mean of three seeds, clean pairs); it passes if its gain over V0 is
# more than noise (the pairs redrawn 10,000 times; the middle 95% above 0). Macro F1 is resampled the same way, deciding nothing.
alls = SCORES[SCORES["pairs"] == "all"].set_index("variant")
tclean = TRUE_SCORES[TRUE_SCORES["pairs"] == "clean"].set_index("variant")
CANDIDATE = "S1" if alls.loc["S1", "destroyed_iou"] >= alls.loc["S2", "destroyed_iou"] else "S2"


def parts_of(v):
    out = []
    for s in SEEDS:
        tp = np.array([CMS[(v, s)][i][3, 3] for i in IMAGES], np.float64)
        un = np.array([CMS[(v, s)][i][3, :].sum() + CMS[(v, s)][i][:, 3].sum() - CMS[(v, s)][i][3, 3] for i in IMAGES], np.float64)
        out.append((tp, un))
    return out


rng = np.random.default_rng(RNG_SEED)
counts = np.stack([np.bincount(r, minlength=len(IMAGES)) for r in rng.integers(0, len(IMAGES), size=(N_RESAMPLES, len(IMAGES)))])
draw_iou = lambda v: np.mean([(counts @ tp) / np.maximum(counts @ un, 1) for tp, un in parts_of(v)], axis=0)
flat = lambda v, s: np.stack([TRUE_CMS[(v, s)][i].ravel() for i in IMAGES]).astype(np.float64)
draw_f1 = lambda v: np.mean([macro_f1_many((counts @ flat(v, s)).reshape(-1, 3, len(LABELS))) for s in SEEDS], axis=0)
res = []
for v in ("S1", "S2"):
    for metric, draw, observed in (("destroyed IoU", draw_iou, alls.loc[v, "destroyed_iou"] - alls.loc["V0", "destroyed_iou"]),
                                   ("macro F1, true buildings", draw_f1, tclean.loc[v, "macro_f1"] - tclean.loc["V0", "macro_f1"])):
        gain = draw(v) - draw("V0")
        res.append({"comparison": f"{v} minus V0", "metric": metric, "observed": round(float(observed), 4),
                    "low_2.5%": round(float(np.percentile(gain, 2.5)), 4), "high_97.5%": round(float(np.percentile(gain, 97.5)), 4),
                    "share_above_0": round(float((gain > 0).mean()), 4), "decides": bool(v == CANDIDATE and metric == "destroyed IoU")})
RES = pd.DataFrame(res)
PASSES = bool(RES.loc[RES["decides"], "low_2.5%"].iloc[0] > 0)
print(RES.to_string(index=False))
verdict = (f"{NAMES[CANDIDATE]} passes: its destroyed IoU gain over V0 is more than noise. Next: step 7b, once on the test "
           "pairs, and retraining the grader on super-resolved crops as a candidate." if PASSES else
           f"{NAMES[CANDIDATE]} doesn't pass: its destroyed IoU gain over V0 isn't clearly above noise. Super-resolution at "
           "grading time doesn't help these graders.")
print(f"\n{'✓' if PASSES else '✗'} {verdict}")

# The share of small buildings, by source (clean pairs)
SIZES["source"] = np.where([bool(XBD.match(Path(n).stem)) for n in SIZES["image"]], "xBD", "other sources")
SIZE_TABLE = pd.DataFrame([{"source": src, "pairs": len(g), "true_buildings": int(g["true_buildings"].sum()),
                            "true_small_share": round(g["true_small"].sum() / max(g["true_buildings"].sum(), 1), 3),
                            "outlines": int(g["outlines"].sum()),
                            "outlines_small_share": round(g["outlines_small"].sum() / max(g["outlines"].sum(), 1), 3)}
                           for src, g in SIZES[SIZES["clean"]].groupby("source")])
print(SIZE_TABLE.to_string(index=False))

# examples.png: each row a small true building; columns: before and after under V0, S1 and S2
if EXAMPLES:
    head = np.full((24, CROP * 6, 3), 255, np.uint8)
    for k, title in enumerate(f"{v} {w}" for v in VARIANTS for w in ("before", "after")):
        cv2.putText(head, title, (k * CROP + 4, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    sheet = [head]
    for e in EXAMPLES:
        row = np.ascontiguousarray(np.hstack([e[v][c:c + 3].transpose(1, 2, 0) for v in VARIANTS for c in (0, 3)]))
        cv2.putText(row, f"true: {e['grade']}", (4, CROP - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1, cv2.LINE_AA)
        sheet.append(row)
    cv2.imwrite(str(OUT_DIR / "examples.png"), cv2.cvtColor(np.vstack(sheet), cv2.COLOR_RGB2BGR))

# Files
SCORES.to_csv(OUT_DIR / "scores.csv", index=False)
TRUE_SCORES.to_csv(OUT_DIR / "true_buildings.csv", index=False)
RES.to_csv(OUT_DIR / "resampling.csv", index=False)
SIZE_TABLE.to_csv(OUT_DIR / "sizes.csv", index=False)
SR_CHECK.to_csv(OUT_DIR / "sr_checks.csv", index=False)
GRADES.to_csv(OUT_DIR / "grades_sr.csv", index=False)
TRUE_GRADES.to_csv(OUT_DIR / "true_grades.csv", index=False)
config = {"task": "R16c step 7", "smoke_test": SMOKE_TEST, "pairs": len(ALL), "clean_pairs": len(IMAGES), "seeds": SEEDS,
          "biases": {s: BIASES[s].tolist() for s in SEEDS}, "crop": CROP, "margin": MARGIN, "min_area": MIN_AREA, "small": SMALL,
          "sr_models": SR_MODELS, "swin2sr_revision": SWIN_REVISION, "realesrgan_sha256": ESR_SHA256, "tile": TILE,
          "tile_pad": TILE_PAD, "sr_devices": [str(d) for d in SR_DEVICES], "sr_photo_seconds": SR_PHOTO_SECONDS,
          "sr_pair_seconds": SR_PAIR_SECONDS, "pairs_super_resolved": n_sr, "cell6_minutes": round(MINUTES, 1),
          "published": PUBLISHED, "score_tolerance": SCORE_TOL, "macro_published": MACRO_PUBLISHED, "macro_tolerance": MACRO_TOL,
          "consistency_db": CONSISTENCY_DB, "resamples": N_RESAMPLES, "candidate": CANDIDATE, "passes": PASSES,
          "torch": torch.__version__, "transformers": transformers.__version__, "opencv": cv2.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")


def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


(OUT_DIR / "README.md").write_text("\n".join([
    f"# R16c step 7: super-resolved crops for the grader{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    f"{len(ALL)} validation pairs, {len(IMAGES)} of them clean; the three graders of R16c steps 5 and 5b; the SAM 3 outlines "
    f"of E3. V0 = crops as before; S1 and S2 = a building whose longest side is under {SMALL} px is cropped from its photos "
    "enlarged 2x by Swin2SR (S1) or Real-ESRGAN (S2). The question, the rule and the expectation are in the notebook's first cell.", "",
    "## Checks", "", md(CHECKS), "",
    "## The SR models on five real photos", "",
    "PSNR in dB against the photo's own centre: `*_psnr` after shrinking it by 2 and enlarging it again (bicubic, S1, S2); "
    f"`*_consistency_psnr` = the model's output shrunk back, against its input (at least {CONSISTENCY_DB} dB).", "",
    md(SR_CHECK), "",
    f"One whole photo: {SR_PHOTO_SECONDS} s; per pair in the run (both photos at once, on {len(SR_DEVICES)} GPU(s)): {SR_PAIR_SECONDS} s.", "",
    f"## Buildings under {SMALL} px (clean pairs)", "", md(SIZE_TABLE), "",
    "## Find-then-grade (mean of three seeds)", "", md(SCORES), "",
    "## The graders on the true buildings (macro F1, mean of three seeds)", "", md(TRUE_SCORES), "",
    "## The rule", "", md(RES), "", f"**Verdict:** {verdict}", "",
    "Files: `scores.csv`, `true_buildings.csv`, `resampling.csv`, `sizes.csv`, `sr_checks.csv`, `grades_sr.csv` (every outline's "
    "grade per version and seed), `true_grades.csv` (every true building's), `examples.png`, `config.json`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, the seven tables, examples.png and config.json to {OUT_DIR}")

                                                                                 check             result  pass
                                    V0 log-probabilities within 0.05 of the saved ones 1.0000 of outlines  True
                             V0 destroyed IoU, seed 42, against 0.3161 (within 0.0005)             0.3161  True
                             V0 destroyed IoU, seed 43, against 0.3051 (within 0.0015)             0.3062  True
                             V0 destroyed IoU, seed 44, against 0.2982 (within 0.0015)             0.2974  True
V0 macro F1 on the true buildings of all pairs, seed 42, against 0.7367 (within 0.003)             0.7367  True
V0 macro F1 on the true buildings of all pairs, seed 43, against 0.7303 (within 0.003)             0.7303  True
V0 macro F1 on the true buildings of all pairs, seed 44, against 0.7541 (within 0.003)             0.7541  True
variant         pairs  n_pairs   miou  destroyed_iou  building_iou  destroyed_iou_s42  destroyed_iou_s43